In [ ]:
! pip install -q sentence-transformers

In [3]:
pd.read_parquet('K:\IMSH\music-recommendations-\data\processed\data.parquet')

<>:1: SyntaxWarning: invalid escape sequence '\I'
<>:1: SyntaxWarning: invalid escape sequence '\I'
C:\Users\Petricia\AppData\Local\Temp\ipykernel_12232\904444267.py:1: SyntaxWarning: invalid escape sequence '\I'
  pd.read_parquet('K:\IMSH\music-recommendations-\data\processed\data.parquet')


,audio,title,artist
0,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02\x03TCO...,Food,AWOL
1,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02=TIT2\x...,Electric Ave,AWOL
2,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02\tTCON\...,This World,AWOL
3,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05&TIT2\x...,Freeway,Kurt Vile
4,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05YTIT2\x...,Spiritual Level,Nicky Cook
...,...,...,...
1225,{'bytes': b'ID3\x04\x00\x00\x00\x00\x04vTIT2\x...,Orgium,Sejayno
1226,{'bytes': b'ID3\x04\x00\x00\x00\x00\x04}TIT2\x...,Vulcan's Hill,Sejayno
1227,{'bytes': b'ID3\x04\x00\x00\x00\x00\x04vTIT2\x...,Garden,Sejayno
1228,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05FTPE1\x...,Reverse Time Apex side 1,Sejayno


In [4]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [5]:
DATA_PATH = "/content/data_fm.parquet"
if DATA_PATH.endswith(".parquet"):
    tracks = pd.read_parquet(DATA_PATH)
else:
    tracks = pd.read_csv(DATA_PATH)

len(tracks), tracks.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/data_fm.parquet'

In [ ]:
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

descriptions = tracks["description"].fillna("").tolist()

embeddings = model.encode(
    descriptions,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

embeddings.shape

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

(909, 768)

In [ ]:
os.makedirs("/content/processed", exist_ok=True)

tracks.to_parquet("/content/processed/data_fm.parquet", index=False)
np.save("/content/processed/data_fm_multi_lannguages_embeddings.npy", embeddings)

In [ ]:
df = pd.read_parquet("/content/processed/data_fm.parquet")

In [ ]:
df

,artist,name,tag,description
0,Harry Styles,Sign of the Times,rock,"""Sign of the Times"" is the debut single from B..."
1,Foo Fighters,Everlong,rock,"""Everlong"" is the second single from Foo Fight..."
2,Goo Goo Dolls,Iris,rock,“Iris” was written for the 1998 film City of A...
3,Paramore,Still Into You,rock,A very upbeat and happy number off their fourt...
4,Jeff Buckley,"Lover, You Should've Come Over",rock,"“Lover, You Should've Come Over“ is track seve..."
...,...,...,...,...
904,Jamiroquai,Virtual Insanity,funk,"""Virtual Insanity"" is a song by the English fu..."
905,Prince,Kiss,funk,"""Kiss"" is a 1986 song by Prince and the Revolu..."
906,Jamiroquai,Space Cowboy,funk,"""Space Cowboy"" is the international lead singl..."
907,Stevie Wonder,Superstition,funk,"""Superstition"" is a popular song written, prod..."


In [ ]:
import numpy as np

def search_tracks(query: str, k: int = 30):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    sims = embeddings @ query_embedding
    top_idx = np.argsort(-sims)[:k]

    result = tracks.iloc[top_idx].copy()
    result["similarity"] = sims[top_idx]

    return result[["artist", "name", "similarity", "description"]]

In [ ]:
query = "melancholic atmospheric rock with emotional vocals"
search_tracks(query, k=5)

,artist,name,similarity,description
657,The Microphones,I Want Wind to Blow,0.597299,The first song on The Microphones' third and m...
300,System of a Down,Lonely Day,0.580023,"""Lonely Day"" is the second single from the 200..."
651,Moby,Natural Blues,0.568926,Moby recalled to Rolling Stone: “Of all the su...
732,Chicane,Don't Give Up,0.566051,"Co-Work between UK Electronical Musician, Comp..."
391,John Mayer,Vultures,0.561486,"""Vultures"" is a song from John Mayer's album C..."


In [ ]:
query = "меланхоличный атмосферный рок с эмоциональным вокалом"
search_tracks(query, k=5)

,artist,name,similarity,description
372,Etta James,I'd Rather Go Blind,0.622489,I'd Rather Go Blind stands as one of the most ...
657,The Microphones,I Want Wind to Blow,0.620308,The first song on The Microphones' third and m...
651,Moby,Natural Blues,0.614035,Moby recalled to Rolling Stone: “Of all the su...
784,Bob Marley & The Wailers,Satisfy My Soul,0.598283,"""Satisfy My Soul"" is a Song from the Kaya tent..."
883,Michael Jackson,Rock With You - Single Version,0.595934,"""Rock with You"" is a smooth disco and funk-inf..."


In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
def retrieve_tracks(query: str, top_n: int = 50):
    query_vec = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    sims = embeddings @ query_vec
    top_idx = np.argsort(-sims)[:top_n]

    return top_idx, sims

In [ ]:
def search_and_rerank(query: str, top_n: int = 50, final_k: int = 5):
    cand_idx, sims = retrieve_tracks(query, top_n=top_n)

    pairs = []
    for idx in cand_idx:
        cand_text = str(tracks.iloc[idx]["description"])
        pairs.append([query, cand_text])

    ce_scores = reranker.predict(pairs)

    result = tracks.iloc[cand_idx].copy()
    result["cosine_sim"] = sims[cand_idx]
    result["cross_score"] = ce_scores

    result = result.sort_values("cross_score", ascending=False).head(final_k)
    return result[["artist", "name", "cosine_sim", "cross_score", "description"]]

In [ ]:
q = "мрачный атмосферный постпанк с гитарой"

print("Топ-5 только по cosine:")
display(search_tracks(q, k=5))

print("Топ-5 после cross-encoder re-rank топ-50:")
display(search_and_rerank(q, top_n=50, final_k=5))

Топ-10 только по cosine:


,artist,name,similarity,description
507,Aphex Twin,Xtal,0.571774,"""Xtal"" is the opening track on Aphex Twin’s al..."
110,OutKast,"So Fresh, So Clean",0.570834,"With a beat courtesy of Organized Noize, 3 Sta..."
634,Air,Cherry Blossom Girl,0.570423,The song is the album's second track. Written ...
520,Aphex Twin,Tha,0.551787,A lot of great 90's electronic music sounded l...
267,Radiohead,Motion Picture Soundtrack,0.546589,"A favorite of Thom's, he expected it to appear..."


Топ-10 после cross-encoder re-rank топ-50:


,artist,name,cosine_sim,cross_score,description
300,System of a Down,Lonely Day,0.491286,-8.421638,"""Lonely Day"" is the second single from the 200..."
305,Disturbed,Stricken,0.483164,-9.159512,"""Stricken"" is the ninth single by American hea..."
657,The Microphones,I Want Wind to Blow,0.519968,-9.261284,The first song on The Microphones' third and m...
302,Avenged Sevenfold,Afterlife,0.518415,-9.347868,"""Afterlife"" is a single by the American hard r..."
887,Wild Cherry,Play That Funky Music,0.508294,-9.426502,"""Play That Funky Music"" (also known as ""Play T..."
